In [ ]:
# M1. 공통 미션의 출발점 — 화면 맨 위 주문서 카드대로 공장을 움직이는 가장 단순한 코드
# 주문서(orders)를 납기 순으로 읽어, 주문마다 색별 수량만큼 펜을 만들고, 상자를 완성품 창고로 보낸 뒤(AGV-3), 트럭에 싣는다(AGV-4).
# 이 코드는 "맞게 움직이지만 느리다": 한 자루를 끝까지 만들고 다음을 투입하므로 주문서 A도 C도 납기를 못 맞춘다.
# 공통 미션은 여기서 출발해, 설비 투자 없이 코드만 고쳐 납기(120·240·400초)를 맞추는 것이다. 정답 코드는 없다.
# 실행하면서 관제 패널의 가동률·역 점유와 보고서를 보고 "시간이 어디서 새는지"를 먼저 찾아라.

def make_pen(p, color):
    """파렛트 p 에 color 펜 한 자루를 만들고 상자에 넣는다. 검사 결과를 돌려준다."""
    feeder.supply(color)                 # ① 호퍼에서 몸체 하나를 출구로
    scara.pick(); scara.place(p)         #    SCARA 가 파렛트에 올린다 → p.color 가 정해진다
    conveyor.move(p, 'P1'); press1.press()   # ② 펜촉
    conveyor.move(p, 'P2'); press2.press()   #    잉크
    conveyor.move(p, 'P3'); press3.press()   #    캡
    conveyor.move(p, 'INSPECT')
    result = robot_a.inspect(p)          # ③ 'OK' 또는 'NG'
    conveyor.move(p, 'SORT')
    robot_b.pick(p)
    box = p.color if result == 'OK' else 'NG'
    if box != 'NG' and boxes.full(box):  # 색 상자(10자루)가 찼으면 AGV-3 로 창고에 먼저 보낸다
        agv.ship(box)
    robot_b.place(box)
    return_conv.send(p)                  # ⑤⑥ 빈 파렛트 회수
    return result

def keep_material(color):
    """호퍼가 바닥나기 전에 자재 랙의 상자를 호퍼에 채운다 (AGV-2). 랙에는 색별 상자 1개(10개)가 있다."""
    if feeder.level(color) <= 3 and material.crates(color) > 0:
        supply_agv.refill(color)

feeder.on()
for o in sorted(orders, key=lambda o: o.due):          # 납기가 빠른 주문부터
    print(f'--- 주문 {o.id}: {o.qty}  납기 {o.due}초 ---')
    for color in ['red', 'blue', 'black']:
        made = 0
        while made < o.qty[color]:                      # 색별 수량만큼, NG 가 나오면 한 번 더
            keep_material(color)
            p = stocker.release()
            if make_pen(p, color) == 'OK':
                made += 1
    for box in ['RED', 'BLUE', 'BLACK']:                # 이 주문의 펜을 창고로 (상자에 있으면 재고가 아니다)
        if boxes.count(box) > 0:
            agv.ship(box)
    outbound_agv.deliver(o)                             # ⑦ AGV-4: 창고 재고에서 이 주문 분량을 납품 트럭에 싣는다
    print(f'주문 {o.id} 납품 · 시각 {clock.now:.0f}초 (납기 {o.due}초{", 지연" if o.late else ""}) · 남은 창고 {warehouse.stock}')
feeder.off()
# 실행 뒤 맨 위 주문 카드가 초록(전량 출고)인지 빨강(미납)인지 보자.
# 이 구조는 8자루(주문서 A)에 155초, 40자루(C)에 약 870초가 걸린다 — 설비가 아니라 '한 자루씩' 코드 구조가 병목이다.
# 다음 단계: 이 코드에서 설비들이 노는 시간이 얼마나 되는지 관제 패널(가동률)로 확인해 보자. 설비 투자는 공통 미션 뒤 팀 미션에서.


In [ ]:
def make_pen(p, color): #(볼펜 한 자루 만드는 함수)
    """파렛트 p 에 color 펜 한 자루를 만들고 상자에 넣는다. 검사 결과를 돌려준다."""
    feeder.supply(color)                 # (지정한 색상의 볼펜 몸체 내보내기)
    scara.pick(); scara.place(p)         # (로봇 팔이 몸체 집기 + 몸체를 팔레트 p에 올리기)
    conveyor.move(p, 'P1'); press1.press()   # 볼펜 조립 펜촉
    conveyor.move(p, 'P2'); press2.press()   #    잉크
    conveyor.move(p, 'P3'); press3.press()   #    캡

    conveyor.move(p, 'INSPECT') # (품질 검사)
    result = robot_a.inspect(p)          #  'OK' 또는 'NG' (정상이면 OK 불량이면 NG)

    conveyor.move(p, 'SORT') # (분류 및 포장)
    robot_b.pick(p)
    box = p.color if result == 'OK' else 'NG'
    if box != 'NG' and boxes.full(box):  # 색 상자(10자루)가 찼으면 AGV-3 로 창고에 먼저 보낸다
        agv.ship(box)
    robot_b.place(box)

    return_conv.send(p)                  # (빈 파렛트 회수)
    return result # (정상인지 불량인지 알려주기)

def keep_material(color): (자재 부족을 방지하는 방수)
    """호퍼가 바닥나기 전에 자재 랙의 상자를 호퍼에 채운다 (AGV-2). 랙에는 색별 상자 1개(10개)가 있다."""
    if feeder.level(color) <= 3 and material.crates(color) > 0:# 남은 몸체가 3개 이하인지 확인
        supply_agv.refill(color)

feeder.on() #(주문서에 따라 생산하는 코드, 보울피더 작동 시작)
for o in sorted(orders, key=lambda o: o.due):          # 주문서를 납기가 빠른 주문부터 순서 정렬
    print(f'--- 주문 {o.id}: {o.qty}  납기 {o.due}초 ---')
    for color in ['red', 'blue', 'black']: # 빨 파 검 순서로 생산
        made = 0
        while made < o.qty[color]:                      # 주문 수량 만족할 때까지 반복, 색별 수량만큼, NG 가 나오면 한 번 더
            keep_material(color)
            p = stocker.release() # 대기 중인 팔렛트 하나 꺼내기
            if make_pen(p, color) == 'OK': # (볼펜 한자루 생산, 정상품일 경우 생산량 1증가(불량이면 증가하지 않음)
                made += 1


    for box in ['RED', 'BLUE', 'BLACK']:                # 이 주문의 펜을 창고로 (상자에 있으면 재고가 아니다)
        if boxes.count(box) > 0:
            agv.ship(box)
    outbound_agv.deliver(o)                             # ⑦ AGV-4: 창고 재고에서 이 주문 분량을 납품 트럭에 싣는다
    print(f'주문 {o.id} 납품 · 시각 {clock.now:.0f}초 (납기 {o.due}초{", 지연" if o.late else ""}) · 남은 창고 {warehouse.stock}')
feeder.off()
# 실행 뒤 맨 위 주문 카드가 초록(전량 출고)인지 빨강(미납)인지 보자.
# 이 구조는 8자루(주문서 A)에 155초, 40자루(C)에 약 870초가 걸린다 — 설비가 아니라 '한 자루씩' 코드 구조가 병목이다.
# 다음 단계: 이 코드에서 설비들이 노는 시간이 얼마나 되는지 관제 패널(가동률)로 확인해 보자. 설비 투자는 공통 미션 뒤 팀 미션에서.


In [ ]:
# M1 문제점
# 1. 순차 생산 방식: 한 자루가 모두 끝나야만 다음 볼펜 생산 시작: 우리 공장은 조립 설비가 3대나 있고 팔렛트도 5장있음
# -> 파이프라인 생산으로 개선 가능

In [ ]:
# B3. 파이프라인 맛보기 — 파렛트 2장을 겹쳐 돌려 빨간 펜 4자루 만들기
# M1 처럼 한 자루를 끝까지 보내고 다음을 투입하면 4자루에 약 78초.
# 설비마다 시간표가 따로 있으므로, 역마다 "다른 파렛트"에 번갈아 명령하면 겹쳐서 돌아 약 53초가 된다.
# 이 예제는 구조만 보여 준다: 색 하나, 주문서 없음, 상자에 넣는 데서 끝.
# 공통 미션에 쓰려면 ① 파렛트를 더 겹치고 ② 주문서(orders)의 색·수량 순서대로 만들고 ③ NG 가 나면 그 색을 다시 만들고
# ④ 호퍼가 비기 전에 보충하고(B2) ⑤ 상자 → 창고 → 트럭(B1 · M1)까지 직접 이어 붙여야 한다. 그것이 과제다.

STEPS = ['LOAD', 'P1', 'P2', 'P3', 'INSPECT', 'SORT']
PRESS = {'P1': press1, 'P2': press2, 'P3': press3}

def work(p):
    """파렛트 p 가 지금 있는 역의 일을 하고 다음 역으로 보낸다. 라인을 떠났으면 True."""
    st = p.station
    if st == 'LOAD':
        feeder.supply('red'); scara.pick(); scara.place(p)
    elif st in PRESS:
        PRESS[st].press()
    elif st == 'INSPECT':
        robot_a.inspect(p)                       # 결과는 p.result 에 남는다
    elif st == 'SORT':
        robot_b.pick(p)
        robot_b.place(p.color if p.result == 'OK' else 'NG')
        return_conv.send(p)
        return True
    conveyor.move(p, STEPS[STEPS.index(st) + 1])
    return False

feeder.on()
todo = 8                       # 만들 펜 수 (8로 수정해보기)
max_pallets = 2 # **지온 추가 코드(최대 동시 투입 팔렛트 )**
on_line = []                   # 라인 위의 파렛트, 앞에 들어간 것부터
while todo > 0 or on_line:
    for p in list(on_line):    # 앞 파렛트부터 처리해야 앞 역이 비어 뒤가 전진한다
        if work(p):
            on_line.remove(p)
    if todo > 0 and len(on_line) < max_pallets and conveyor.at('LOAD') is None:   # 2장까지만 겹친다 (숫자를 바꿔 보자) **팔렛트 투입 시점과 동시 투입 수량을 결정하는 중요한 코드**
        on_line.append(stocker.release())
        todo -= 1
feeder.off()
print('종료 %.1f초 / 상자 %s' % (clock.now, boxes.counts))

# max_pallets 값을 1~5로 바꿔가면서 실험해보기
# 무조건 팔렛트가 많을 수록 생산 효율이 높아지는 것이 아님.
# 공정 내 대기나 병목현상이 발생할 수 있수므로 실험해서 적정 수량 결정

In [ ]:
# B3: 여러 팔렛트를 투입하여 서로 다른 공정에서 작업을 겹쳐 진행함.

# max 5는 시각: 116.7 / ok: 8 / ng: 0
# max 4는 시각: 118.7 / ok: 8 / ng: 0
# max 3는 시각: 126.1 / ok: 8 / ng: 0
# max 2는 시각: 143.2 / ok: 8 / ng: 0
# max 1는 시각: 197.0 / ok: 8 / ng: 0

# 팔렛트를 5장 동시에 활용하는 것이 가장 효율적이라는 결과가 나타났지만 한계효과 체감 존재


# 목표: M1의 주문·물류 관리 기능과 B3의 파이프라인 생산 방식을 결합하여 납기를 충족하는 것.

In [ ]:
# B1. 상자 → AGV-3(완성품) → 완성품 운반
# 색 상자는 10자루가 차면 더 넣을 수 없다. agv.ship(상자)를 불러야 창고 재고가 된다.
# 빨간 펜 12자루를 만들면서 상자가 차는 순간을 코드가 처리하는 방법을 본다.
# (호퍼에는 빨강 몸체 10개 + 자재 창고에 상자 1개(10개)가 있어 12자루까지는 자재 걱정이 없다 → B2·B3)

def make_pen(p, color):
    feeder.supply(color); scara.pick(); scara.place(p)
    for st, pr in zip(['P1', 'P2', 'P3'], [press1, press2, press3]):
        conveyor.move(p, st); pr.press()
    conveyor.move(p, 'INSPECT')
    r = robot_a.inspect(p)
    conveyor.move(p, 'SORT')
    robot_b.pick(p)
    box = p.color if r == 'OK' else 'NG'
    if box != 'NG' and boxes.full(box):        # 상자가 가득 찼다 → AGV-3 먼저
        n = agv.ship(box)
        print(f'  {box} 상자 {n}자루 → 완성품 창고로 출발 (AGV-3)')
    robot_b.place(box)
    return_conv.send(p)
    return r

feeder.on()
supply_agv.refill('red')                       # 자재 창고의 빨강 상자 1개를 호퍼에 미리 부어 둔다 (10 → 20개)
for i in range(12):
    p = stocker.release()
    r = make_pen(p, 'red')
    print(f'{i + 1:2d}번째 → {r}   상자 {boxes.count("RED")}/{boxes.capacity}')
feeder.off()

if boxes.count('RED') > 0:                     # 다 안 찬 상자도 마지막에는 보낸다
    agv.ship('RED')
print('창고 재고:', warehouse.stock, '/ 전체 종료 %.1f초' % clock.now)


In [ ]:
# B2. 자재 보충 / 자재 창고 랙 → AGV-2(자재) → 피더 호퍼
# 호퍼에는 색별로 몸체 10개가 들어 있다 (용량 20). 비면 feeder.supply() 가 오류를 낸다.
# AGV-2 는 랙에 있는 상자만 나를 수 있다 (왕복 16초, 호퍼에 들어가는 건 출발 11초 뒤) → '미리' 불러야 라인이 기다리지 않는다.
# 특히 B2에서 중요한 점은 자재 보충에 왕복 16초, 실제 호퍼에 도착하기까지 11초가 걸린다는 거야. 따라서 자재가 완전히 소진되기 전에 보충해야 생산라인의 대기를 줄일 수 있어.

def make_pen(p, color):
    feeder.supply(color); scara.pick(); scara.place(p)
    for st, pr in zip(['P1', 'P2', 'P3'], [press1, press2, press3]):
        conveyor.move(p, st); pr.press()
    conveyor.move(p, 'INSPECT')
    r = robot_a.inspect(p)
    conveyor.move(p, 'SORT')
    robot_b.pick(p)
    box = p.color if r == 'OK' else 'NG'
    if box != 'NG' and boxes.full(box):          # B1 에서 배운 것
        agv.ship(box)
    robot_b.place(box); return_conv.send(p)
    return r

feeder.on()
print('시작 호퍼:', feeder.levels, '/ 자재 랙:', material.stock)

# 1) 호퍼가 빌 때까지 기다렸다가 부르면? → 라인이 11초 동안 멈춘다
for i in range(10):
    p = stocker.release(); make_pen(p, 'red')
print('10자루 뒤 빨강 호퍼:', feeder.level('red'), '/ 시각 %.1f초' % clock.now)
try:
    feeder.supply('red')
except FactoryError as e:
    print('오류 →', e)
t = clock.now
supply_agv.refill('red')                       # 랙의 빨강 상자 1개 → 호퍼. 지금 부르면 11초 뒤에야 들어온다
p = stocker.release(); make_pen(p, 'red')
print('보충 후 첫 자루까지 %.1f초 (평소보다 길다)' % (clock.now - t))
if boxes.full('RED'):
    agv.ship('RED')

# 2) 잔량을 보고 미리 부르면 → 기다리지 않는다. 이번엔 파랑으로 (랙에는 색별 상자 1개가 있어 색마다 한 번 보충할 수 있다)
for i in range(8):
    if feeder.level('blue') <= 3 and supply_agv.busy_until <= clock.now:
        supply_agv.refill('blue')                                # 랙 → 호퍼
        print('  미리 보충 요청 (잔량 %d, 시각 %.1f초)' % (feeder.level('blue'), clock.now))
    p = stocker.release(); make_pen(p, 'blue')
feeder.off()
print('끝 호퍼:', feeder.levels, '/ 자재 랙:', material.stock, '/ 상자:', boxes.counts, '/ 시각 %.1f초' % clock.now)


In [ ]:
# 공통 미션 통합 코드(B3파이프라인 구조를 전체 설계로 도입 후 B2와 B1을 합침 -> 아마도..?)

STEPS = ['LOAD', 'P1', 'P2', 'P3', 'INSPECT', 'SORT']
PRESS = {'P1': press1, 'P2': press2, 'P3': press3}
COLORS = ['red', 'blue', 'black']

max_pallets = 5

# 자재 사전 보충
def keep_material(color):
    if feeder.level(color) <= 3 and material.crates(color) > 0:
        if supply_agv.busy_until <= clock.now:
            supply_agv.refill(color)

# 팔렛트의 현재 공정 처리
def work(p, color):
    st = p.station

    if st == 'LOAD':
        keep_material(color)
        feeder.supply(color)
        scara.pick()
        scara.place(p)

    elif st in PRESS:
        PRESS[st].press()

    elif st == 'INSPECT':
        robot_a.inspect(p)

    elif st == 'SORT':
        robot_b.pick(p)
        box = p.color if p.result == 'OK' else 'NG'

        if box != 'NG' and boxes.full(box):
            agv.ship(box)

        robot_b.place(box)
        return_conv.send(p)
        return True

    conveyor.move(p, STEPS[STEPS.index(st) + 1])
    return False


feeder.on()

# 납기가 빠른 주문부터 처리
for o in sorted(orders, key=lambda o: o.due):

    queue = []
    on_line = []

    # 주문서에 따른 생산 대기열 생성
    for color in COLORS:
        queue.extend([color] * o.qty[color]) # 생산 대기열(색상이 바뀌어도 연속적으로 생산 가능)

    # 파이프라인 생산
    while queue or on_line:

        for p, color in list(on_line):
            if work(p, color):
                on_line.remove((p, color))

                # 불량품은 동일 색상으로 추가 생산
                if p.result == 'NG':
                    queue.append(color)

        # 새로운 팔렛트 투입
        if queue and len(on_line) < max_pallets:
            if conveyor.at('LOAD') is None:
                color = queue.pop(0)
                p = stocker.release()
                on_line.append((p, color))

    # 완성품 창고로 운반
    for box in ['RED', 'BLUE', 'BLACK']:
        if boxes.count(box) > 0:
            agv.ship(box)

    # 주문 납품
    outbound_agv.deliver(o)

    print(f'주문 {o.id} / 완료 {clock.now:.1f}초 / 납기 {o.due}초')

feeder.off()
print(f'전체 종료: {clock.now:.1f}초')

In [ ]:
# 공통 미션 통합 코드 + 팀 미션(1)
# 가설: 자재 보충 시점을 앞당기면 자재 부족으로 발생할 수 있는 생산 대기 시작 감소할 수 있을 것임

STEPS = ['LOAD', 'P1', 'P2', 'P3', 'INSPECT', 'SORT']
PRESS = {'P1': press1, 'P2': press2, 'P3': press3}
COLORS = ['red', 'blue', 'black']

max_pallets = 5
refill_level = 3 # **자재 보충 기준 잔량**

# 자재 사전 보충
def keep_material(color):
    if feeder.level(color) <= refill_level and material.crates(color) > 0:
        if supply_agv.busy_until <= clock.now:
            supply_agv.refill(color)

# 팔렛트의 현재 공정 처리
def work(p, color):
    st = p.station

    if st == 'LOAD':
        keep_material(color)
        feeder.supply(color)
        scara.pick()
        scara.place(p)

    elif st in PRESS:
        PRESS[st].press()

    elif st == 'INSPECT':
        robot_a.inspect(p)

    elif st == 'SORT':
        robot_b.pick(p)
        box = p.color if p.result == 'OK' else 'NG'

        if box != 'NG' and boxes.full(box):
            agv.ship(box)

        robot_b.place(box)
        return_conv.send(p)
        return True

    conveyor.move(p, STEPS[STEPS.index(st) + 1])
    return False


feeder.on()

# 납기가 빠른 주문부터 처리
for o in sorted(orders, key=lambda o: o.due):

    queue = []
    on_line = []

    # 주문서에 따른 생산 대기열 생성
    for color in COLORS:
        queue.extend([color] * o.qty[color]) # 생산 대기열(색상이 바뀌어도 연속적으로 생산 가능)

    # 파이프라인 생산
    while queue or on_line:

        for p, color in list(on_line):
            if work(p, color):
                on_line.remove((p, color))

                # 불량품은 동일 색상으로 추가 생산
                if p.result == 'NG':
                    queue.append(color)

        # 새로운 팔렛트 투입
        if queue and len(on_line) < max_pallets:
            if conveyor.at('LOAD') is None:
                color = queue.pop(0)
                p = stocker.release()
                on_line.append((p, color))

    # 완성품 창고로 운반
    for box in ['RED', 'BLUE', 'BLACK']:
        if boxes.count(box) > 0:
            agv.ship(box)

    # 주문 납품
    outbound_agv.deliver(o)

    print(f'주문 {o.id} / 완료 {clock.now:.1f}초 / 납기 {o.due}초')

feeder.off()
print(f'전체 종료: {clock.now:.1f}초')

In [ ]:
4 / 372.3
5 / 372.7
6 / 372.9
7 / 373
8 / 372.8
9 / 372.8
10 / 372.8

자재 보충 시점 변경은 개선 효과가 없는 것으로 판단하고 기존 기준인 3개를 유지

In [ ]:
# 공통 미션 통합 코드 + 팀미션(2)
# 색상별 생산 순서 최적화. 이번에는 기존 코드의 고정된 생산 순서를 개선해 보자.

STEPS = ['LOAD', 'P1', 'P2', 'P3', 'INSPECT', 'SORT']
PRESS = {'P1': press1, 'P2': press2, 'P3': press3}
COLORS = ['red', 'blue', 'black'] # 색 순서 바꿔보기

max_pallets = 5

# 자재 사전 보충
def keep_material(color):
    if feeder.level(color) <= 3 and material.crates(color) > 0:
        if supply_agv.busy_until <= clock.now:
            supply_agv.refill(color)

# 팔렛트의 현재 공정 처리
def work(p, color):
    st = p.station

    if st == 'LOAD':
        keep_material(color)
        feeder.supply(color)
        scara.pick()
        scara.place(p)

    elif st in PRESS:
        PRESS[st].press()

    elif st == 'INSPECT':
        robot_a.inspect(p)

    elif st == 'SORT':
        robot_b.pick(p)
        box = p.color if p.result == 'OK' else 'NG'

        if box != 'NG' and boxes.full(box):
            agv.ship(box)

        robot_b.place(box)
        return_conv.send(p)
        return True

    conveyor.move(p, STEPS[STEPS.index(st) + 1])
    return False


feeder.on()

# 납기가 빠른 주문부터 처리
for o in sorted(orders, key=lambda o: o.due):

    queue = []
    on_line = []

    # 주문서에 따른 생산 대기열 생성
    for color in COLORS:
        queue.extend([color] * o.qty[color]) # 생산 대기열(색상이 바뀌어도 연속적으로 생산 가능)

    # 파이프라인 생산
    while queue or on_line:

        for p, color in list(on_line):
            if work(p, color):
                on_line.remove((p, color))

                # 불량품은 동일 색상으로 추가 생산
                if p.result == 'NG':
                    queue.append(color)

        # 새로운 팔렛트 투입
        if queue and len(on_line) < max_pallets:
            if conveyor.at('LOAD') is None:
                color = queue.pop(0)
                p = stocker.release()
                on_line.append((p, color))

    # 완성품 창고로 운반
    for box in ['RED', 'BLUE', 'BLACK']:
        if boxes.count(box) > 0:
            agv.ship(box)

    # 주문 납품
    outbound_agv.deliver(o)

    print(f'주문 {o.id} / 완료 {clock.now:.1f}초 / 납기 {o.due}초')

feeder.off()
print(f'전체 종료: {clock.now:.1f}초')

In [ ]:
(기존)	red, blue, black	371.8초
2	red, black, blue	359.9초 / 납기 달성 / OK:40, NG:10 # 채택
3	blue, red, black	386.2초 / 납기 달성 / OK:40, NG:10
4	blue, black, red	371.2초 / 납기 달성 / OK:40, NG:10
5 black, red, blue	386.2초 / 납기 달성 / OK:40, NG:10
6 black, blue, red	372.6초 / 납기 달성 / OK:40, NG:10


In [ ]:
# 공통 미션 통합 코드 + 팀미션(3)
# 물류 운반 시점 최적화
# 개선 가설: 상자가 가득 차는 즉시 운반을 요청

STEPS = ['LOAD', 'P1', 'P2', 'P3', 'INSPECT', 'SORT']
PRESS = {'P1': press1, 'P2': press2, 'P3': press3}
COLORS = ['red', 'black', 'blue']

max_pallets = 5

# 자재 사전 보충
def keep_material(color):
    if feeder.level(color) <= 3 and material.crates(color) > 0:
        if supply_agv.busy_until <= clock.now:
            supply_agv.refill(color)

# 팔렛트의 현재 공정 처리
def work(p, color):
    st = p.station

    if st == 'LOAD':
        keep_material(color)
        feeder.supply(color)
        scara.pick()
        scara.place(p)

    elif st in PRESS:
        PRESS[st].press()

    elif st == 'INSPECT':
        robot_a.inspect(p)

    elif st == 'SORT':
        robot_b.pick(p)
        box = p.color if p.result == 'OK' else 'NG'

        robot_b.place(box) # 기존코드를 순서만 변경
        #10번째 볼펜 넣자마자 운반 요청하는 방식

        if box != 'NG' and boxes.full(box): # 기존 코드 문제점: 현재는 상자가 가득 차더라도 다음 양품을 넣으려는 시점까지 운반을 요청하지 않음
            agv.ship(box)


        return_conv.send(p)
        return True

    conveyor.move(p, STEPS[STEPS.index(st) + 1])
    return False


feeder.on()

# 납기가 빠른 주문부터 처리
for o in sorted(orders, key=lambda o: o.due):

    queue = []
    on_line = []

    # 주문서에 따른 생산 대기열 생성
    for color in COLORS:
        queue.extend([color] * o.qty[color]) # 생산 대기열(색상이 바뀌어도 연속적으로 생산 가능)

    # 파이프라인 생산
    while queue or on_line:

        for p, color in list(on_line):
            if work(p, color):
                on_line.remove((p, color))

                # 불량품은 동일 색상으로 추가 생산
                if p.result == 'NG':
                    queue.append(color)

        # 새로운 팔렛트 투입
        if queue and len(on_line) < max_pallets:
            if conveyor.at('LOAD') is None:
                color = queue.pop(0)
                p = stocker.release()
                on_line.append((p, color))

    # 완성품 창고로 운반
    for box in ['RED', 'BLUE', 'BLACK']:
        if boxes.count(box) > 0:
            agv.ship(box)

    # 주문 납품
    outbound_agv.deliver(o)

    print(f'주문 {o.id} / 완료 {clock.now:.1f}초 / 납기 {o.due}초')

feeder.off()
print(f'전체 종료: {clock.now:.1f}초')

In [ ]:
# 미채택. 주문별 색상 수량이 최대 8개이고 주문이 끝날 때마다 상자를 비우기 때문에, 상자가 가득 차는 상황 자체가 발생하지 않는 것으로 보임.

In [ ]:
# 공통 미션 통합 코드 + 팀미션(4)
# 색상별 생산 순서 최적화. 이번에는 기존 코드의 고정된 생산 순서를 개선해 보자.

STEPS = ['LOAD', 'P1', 'P2', 'P3', 'INSPECT', 'SORT']
PRESS = {'P1': press1, 'P2': press2, 'P3': press3}
COLORS = ['red', 'black', 'blue'] # 색 순서 바꿔보기

max_pallets = 5

# 자재 사전 보충
def keep_material(color):
    if feeder.level(color) <= 3 and material.crates(color) > 0:
        if supply_agv.busy_until <= clock.now:
            supply_agv.refill(color)

# 팔렛트의 현재 공정 처리
def work(p, color):
    st = p.station

    if st == 'LOAD':
        keep_material(color)
        feeder.supply(color)
        scara.pick()
        scara.place(p)

    elif st in PRESS:
        PRESS[st].press()

    elif st == 'INSPECT':
        robot_a.inspect(p)

    elif st == 'SORT':
        robot_b.pick(p)
        box = p.color if p.result == 'OK' else 'NG'

        if box != 'NG' and boxes.full(box):
            agv.ship(box)

        robot_b.place(box)
        return_conv.send(p)
        return True

    conveyor.move(p, STEPS[STEPS.index(st) + 1])
    return False


feeder.on()

# 납기가 빠른 주문부터 처리
for o in sorted(orders, key=lambda o: o.due):

    queue = []
    on_line = []

    # 주문서에 따른 생산 대기열 생성
    for color in COLORS:
        queue.extend([color] * o.qty[color]) # 생산 대기열(색상이 바뀌어도 연속적으로 생산 가능)

    # 파이프라인 생산
    while queue or on_line:

        for p, color in list(on_line):
            if work(p, color):
                on_line.remove((p, color))

                # 불량품은 동일 색상으로 추가 생산
                if p.result == 'NG':
                queue.append(color)

                elif boxes.count(color.upper()) == o.qty[color]:
                    if agv.busy_until <= clock.now:
                       agv.ship(color.upper())

        # 새로운 팔렛트 투입
        if queue and len(on_line) < max_pallets:
            if conveyor.at('LOAD') is None:
                color = queue.pop(0)
                p = stocker.release()
                on_line.append((p, color))

    # 완성품 창고로 운반
    for box in ['RED', 'BLUE', 'BLACK']:
        if boxes.count(box) > 0:
            agv.ship(box)

    # 주문 납품
    outbound_agv.deliver(o)

    print(f'주문 {o.id} / 완료 {clock.now:.1f}초 / 납기 {o.due}초')

feeder.off()
print(f'전체 종료: {clock.now:.1f}초')